In [11]:
# ============================================================
# 02_preprocessing.ipynb — NLP Preprocessing Pipeline
# ============================================================

In [12]:
import pandas as pd
import re
import nltk
import csv
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [13]:
nltk.download('stopwords')
nltk.download('wordnet')

df = pd.read_csv("../data/raw/Hotel_Reviews.csv")
print("Original shape:", df.shape)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# --- Pipeline for BoW / TF-IDF ---
def preprocess(text):
    text = re.sub(r'<.*?>', '', str(text))
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    tokens = [w for w in tokens if len(w) > 2]  # FIX: drop single-char / 2-char noise tokens
    return " ".join(tokens)

# --- Pipeline for DistilBERT (keep full context, no stopword removal) ---
def preprocess_bert(text):
    text = re.sub(r'<.*?>', '', str(text))   # strip HTML only
    text = re.sub(r'\s+', ' ', text)          # normalise whitespace
    return text.strip()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\windows\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\windows\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Original shape: (515738, 17)


In [14]:
neg = df[~df['Negative_Review'].str.strip().isin(['No Negative', ''])][
    ['Negative_Review', 'Hotel_Name', 'Average_Score', 'Reviewer_Score']
].copy()
neg.rename(columns={'Negative_Review': 'review_text'}, inplace=True)
neg['label'] = 'Negative'

In [15]:
pos = df[~df['Positive_Review'].str.strip().isin(['No Positive', ''])][
    ['Positive_Review', 'Hotel_Name', 'Average_Score', 'Reviewer_Score']
].copy()
pos.rename(columns={'Positive_Review': 'review_text'}, inplace=True)
pos['label'] = 'Positive'

In [16]:
neg['_idx'] = neg.index
pos['_idx'] = pos.index

combined = pd.concat([neg, pos], ignore_index=True)
combined = combined.sort_values('_idx').drop(columns='_idx').reset_index(drop=True)

print("After unpivot:", combined.shape)
print(combined['label'].value_counts())

After unpivot: (866608, 5)
label
Positive    479609
Negative    386999
Name: count, dtype: int64


In [17]:
combined['clean_text'] = combined['review_text'].apply(preprocess)
combined['bert_text']  = combined['review_text'].apply(preprocess_bert)

In [18]:
# Drop rows that are empty after cleaning
combined = combined[combined['clean_text'].str.strip() != '']
combined = combined.dropna(subset=['clean_text'])

# FIX: drop reviews with fewer than 3 meaningful tokens
combined = combined[combined['clean_text'].str.split().str.len() >= 3]

In [19]:
df_final = combined[['clean_text', 'bert_text', 'label',
                      'Hotel_Name', 'Average_Score', 'Reviewer_Score']].copy()

print("\nFinal shape:", df_final.shape)
print(df_final['label'].value_counts())
print("\nSample (first 6 rows):")
print(df_final[['clean_text', 'bert_text', 'label', 'Reviewer_Score']].head(6))


Final shape: (709313, 6)
label
Positive    399453
Negative    309860
Name: count, dtype: int64

Sample (first 6 rows):
                                          clean_text  \
0  angry made post available via possible site us...   
1                       park outside hotel beautiful   
2  real complaint hotel great great location surr...   
3  location good staff cute hotel breakfast range...   
4  room nice elderly bit difficult room two story...   
5  great location nice surroundings bar restauran...   

                                           bert_text     label  Reviewer_Score  
0  I am so angry that i made this post available ...  Negative             2.9  
1   Only the park outside of the hotel was beautiful  Positive             2.9  
2  No real complaints the hotel was great great l...  Positive             7.5  
3  Location was good and staff were ok It is cute...  Positive             7.1  
4  Rooms are nice but for elderly a bit difficult...  Negative             7.1  
5

In [20]:
df_final.to_csv(
    '../data/processed/clean_reviews.csv',
    index=False,
    sep=',',
    encoding='utf-8',
    quoting=csv.QUOTE_ALL
)
print("\n✅ Saved!")


✅ Saved!
